# 02 — Results Overview

Visualize performance across all model/dataset combinations: heatmaps, bar charts, summary tables.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

# Load aggregated results
test_df = pd.read_csv('../results/aggregated/test_results.csv')
fold_df = pd.read_csv('../results/aggregated/fold_results.csv')
test_df.head()

In [ ]:
# Performance heatmap — Classification (AUC / Accuracy)
clf_df = test_df[test_df['task_type'].isin(['binary', 'multiclass'])].copy()

# Use AUC for binary, accuracy for multiclass
clf_df['primary_metric'] = clf_df.apply(
    lambda r: r.get('roc_auc', r.get('accuracy', np.nan))
    if r['task_type'] == 'binary' else r.get('accuracy', np.nan),
    axis=1
)

pivot_clf = clf_df.pivot(index='dataset', columns='model', values='primary_metric')

plt.figure(figsize=(12, 5))
sns.heatmap(pivot_clf, annot=True, fmt='.3f', cmap='YlGn', linewidths=0.5)
plt.title('Classification Performance (AUC for binary, Accuracy for multiclass)')
plt.tight_layout()
plt.savefig('../results/figures/heatmap_classification.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Performance heatmap — Regression (RMSE)
reg_df = test_df[test_df['task_type'] == 'regression'].copy()
pivot_reg = reg_df.pivot(index='dataset', columns='model', values='rmse')

plt.figure(figsize=(12, 5))
sns.heatmap(pivot_reg, annot=True, fmt='.3f', cmap='YlOrRd_r', linewidths=0.5)
plt.title('Regression Performance (RMSE — lower is better)')
plt.tight_layout()
plt.savefig('../results/figures/heatmap_regression.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-validation variance: box plots per model
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Classification
clf_folds = fold_df[fold_df['task_type'].isin(['binary', 'multiclass'])]
if 'accuracy' in clf_folds.columns:
    sns.boxplot(data=clf_folds, x='model', y='accuracy', ax=axes[0])
    axes[0].set_title('Classification — Accuracy across CV Folds')
    axes[0].tick_params(axis='x', rotation=45)

# Regression
reg_folds = fold_df[fold_df['task_type'] == 'regression']
if 'rmse' in reg_folds.columns:
    sns.boxplot(data=reg_folds, x='model', y='rmse', ax=axes[1])
    axes[1].set_title('Regression — RMSE across CV Folds')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/figures/cv_variance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table: mean ± std across folds, per dataset
for task in ['binary', 'multiclass', 'regression']:
    subset = fold_df[fold_df['task_type'] == task]
    if subset.empty:
        continue

    metric = 'roc_auc' if task == 'binary' else ('accuracy' if task == 'multiclass' else 'rmse')
    if metric not in subset.columns:
        continue

    summary = subset.groupby(['dataset', 'model'])[metric].agg(['mean', 'std'])
    summary['display'] = summary.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
    pivot = summary['display'].unstack('model')

    print(f"\n=== {task.upper()} — {metric} (mean ± std) ===")
    display(pivot)